## <center> **Part 1: Recommend classic rock songs and build Neural Networks only with numpy** <center>
### <center> **Mia Mac** <center>

### **I. Importing Data and Libraries**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
df = pd.read_csv('/Users/macminhanh/Documents/UChicago/Machine Learning II/Assignment 3/radio_songs.csv')

### **II. EDA**

#### **1. Data Overview**

In [9]:
df.shape

(100, 285)

In [10]:
df.head()

,user,abba,ac/dc,adam green,aerosmith,afi,air,alanis morissette,alexisonfire,alicia keys,...,timbaland,tom waits,tool,tori amos,travis,trivium,u2,underoath,volbeat,yann tiersen
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,33,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,42,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,51,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,62,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Columns: 285 entries, user to yann tiersen
dtypes: int64(285)
memory usage: 222.8 KB


#### **2. Missing Values**

In [12]:
print(df.isnull().sum().sum())

0


#### **3. Sparsity**

In [15]:
(df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100

np.float64(0.0)

#### **4. Check if user 1060 exists as a row**
Even a data looks clean, a useID could still be missing from the indez if the csv was filtered or formatted differenlty. 

In [20]:
df = df.set_index('user')
1606 in df.index

True

In [21]:
# check user 1606's listening history
print(f"\nSongs user 1606 listened to: {(df.loc[1606] > 0).sum()}")
print(df.loc[1606][df.loc[1606] > 0])  # list the actual artists


Songs user 1606 listened to: 4
abba             1
elvis presley    1
frank sinatra    1
the beatles      1
Name: 1606, dtype: int64


We skip distribution plots, outlier analysis, and correlation heatmaps here since CF algorithm handels the math.
We also skip detailed gener and song analysis since it is not needed for cosine similarity. 

### **III. Collaborative Filtering**

#### <center> **Item-Item Collaborative Filtering** </center>
#### **A. Recommend 10 artists to users who have listened to u2 and pink floyd**

##### **1. Making sure the dataframe is in the format of artists become rows, users become columns**
- For item-item CF, we compare artist against each other.
- Cosine similarity works on rows, so each artist needs to be its own row.

In [22]:
df_T = df.T
print(df_T.shape)

(284, 100)


In [23]:
df_T.head()

user,1,33,42,51,62,75,130,141,144,150,...,1521,1530,1536,1545,1549,1566,1586,1589,1601,1606
abba,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
ac/dc,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
adam green,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aerosmith,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
afi,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


##### **2. Compute cosine similarity between all artists**
- Creating a full artist x artist similarity matrix
- similarity = 1 - cosine_distance
- similarity = 1 -> identical listening patterns
- similarity = 0 -> completely different

In [24]:
item_sim_matrix = pd.DataFrame(
    cosine_similarity(df_T),
    index=df_T.index,
    columns=df_T.index
)
print(item_sim_matrix.shape)

(284, 284)


In [25]:
item_sim_matrix.head()

,abba,ac/dc,adam green,aerosmith,afi,air,alanis morissette,alexisonfire,alicia keys,all that remains,...,timbaland,tom waits,tool,tori amos,travis,trivium,u2,underoath,volbeat,yann tiersen
abba,1.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
ac/dc,0.0,1.000000,0.0,0.0,0.223607,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.223607,0.0,0.2,0.0
adam green,0.0,0.000000,1.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
aerosmith,0.0,0.000000,0.0,1.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
afi,0.0,0.223607,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


##### **3. Looking at similarity scores for 'u2' and 'pink floyd'**
- Finding which artists are most similar to these two specifically

In [33]:
u2_sim = item_sim_matrix['u2'].sort_values(ascending=False)
pf_sim = item_sim_matrix['pink floyd'].sort_values(ascending=False)

print("Top 20 artists similar to u2:")
print(u2_sim[1:20]) 

print("Top 20 artists similar to pink floyd:")
print(pf_sim[1:20]) 

Top 20 artists similar to u2:
robbie williams            0.500000
misfits                    0.500000
green day                  0.433013
depeche mode               0.408248
peter fox                  0.377964
kelly clarkson             0.353553
johnny cash                0.353553
madonna                    0.353553
dire straits               0.353553
enter shikari              0.353553
moby                       0.301511
foo fighters               0.301511
queens of the stone age    0.288675
nickelback                 0.288675
lady gaga                  0.288675
nirvana                    0.288675
the chemical brothers      0.250000
morcheeba                  0.250000
guano apes                 0.223607
Name: u2, dtype: float64
Top 20 artists similar to pink floyd:
genesis                     0.577350
funeral for a friend        0.408248
queen                       0.408248
hans zimmer                 0.408248
led zeppelin                0.408248
sonic syndicate             0.408248
d

- For U2 listeners: Robbie Willliams and Misfits are most similar (0.5), meaning the same users who listen to u2 also tend to listern to them.
- For Pink Floyd listeners: Genesis is the strongest match.


##### **4. Combine similarities from both u2 and pink floyd**
- Looking at users who listened to both artists.
- Average thier similarity scores to get a combined ranking.

In [29]:
combined_sim = (item_sim_matrix['u2'] + item_sim_matrix['pink floyd']) / 2

##### **5. Remove u2 and pink floyd themselves from the recommendation because we don't want to recommend an artist the user already listens to.**

In [30]:
combined_sim = combined_sim.drop(['u2', 'pink floyd'])

##### **6. Top 10 recommendations**

In [31]:
top10_recommendations = combined_sim.sort_values(ascending=False).head(10)

print("TOP 10 RECOMMENDED ARTISTS FOR U2 + PINK FLOYD LISTENERS")
print(top10_recommendations)

TOP 10 RECOMMENDED ARTISTS FOR U2 + PINK FLOYD LISTENERS
robbie williams    0.394338
genesis            0.288675
johnny cash        0.278839
misfits            0.250000
audioslave         0.240903
foo fighters       0.237795
pearl jam          0.219913
green day          0.216506
depeche mode       0.204124
david bowie        0.204124
dtype: float64


The most recommended artist is Robbie Williams, which is pretty obvious since he is in both of the top 20 artist are most similar to U2 and Pink Floyd.

#### <center> **User-User Collaborative Filtering** <center> 
#### **B. Find user most similar to user 1606.**

##### **1. Compute user-user similarity matrix**
- Computing how similar every user is to every other user based on their listening history

In [35]:
user_sim_matrix = pd.DataFrame(
    cosine_similarity(df),
    index=df.index,
    columns=df.index
)

print(f"Shape: {user_sim_matrix.shape}")
print(user_sim_matrix.iloc[:5, :5])

Shape: (100, 100)
user        1         33        42        51        62
user                                                  
1     1.000000  0.061546  0.000000  0.000000  0.083624
33    0.061546  1.000000  0.077152  0.247537  0.226455
42    0.000000  0.077152  1.000000  0.000000  0.000000
51    0.000000  0.247537  0.000000  1.000000  0.336336
62    0.083624  0.226455  0.000000  0.336336  1.000000


##### **2. Find the most similar user to user 1060**
- Looking at user 1060's row in the similarity matrix and finds which other user has the highest similatiy score

In [36]:
# Sort from highest to lowest similarity
user_1606_sim = user_sim_matrix[1606].sort_values(ascending=False)

In [37]:
# Print top 5 most similar users
print("Top 5 most similar users to user 1606:")
print(user_1606_sim.head(6))  

Top 5 most similar users to user 1606:
user
1606    1.000000
1144    0.277350
144     0.223607
1334    0.150756
1509    0.144338
890     0.129099
Name: 1606, dtype: float64


In [39]:
# Identify the single most similar user
most_similar_user = user_1606_sim.index[1]  
most_similar_score = user_1606_sim.iloc[1]

In [41]:
print(f"Most similar user to 1606: User {most_similar_user}")
print(f"Similarity score: {most_similar_score:.4f}")

Most similar user to 1606: User 1144
Similarity score: 0.2774


- The most similar user to 1606 is user 1144 with a similarity score of 0.2774.
- The score is not super high. This makes sense since use 1606 only listened to 4 artists (based on our EDA).

##### **3. Get all the artists that User 1144 has listened to**


In [44]:
similar_user_artists = df.loc[most_similar_user]

# Filter only artists they actually listened to 
similar_user_listened = similar_user_artists[similar_user_artists > 0].index.tolist()

print(f"User 1144 has listened to {len(similar_user_listened)} artists:")
print(similar_user_listened)

User 1144 has listened to 13 artists:
['beastie boys', 'bob dylan', 'bob marley & the wailers', 'david bowie', 'elvis presley', 'eric clapton', 'johnny cash', 'pearl jam', 'pink floyd', 'the beatles', 'the doors', 'the rolling stones', 'tom waits']


##### **4. Filter out artists user 1606 already know**
- We don't want to recommend artists that user alreayd knew.

In [45]:
user_1606_listened = df.loc[1606]
user_1606_listened = user_1606_listened[user_1606_listened > 0].index.tolist()
user_1606_listened 

['abba', 'elvis presley', 'frank sinatra', 'the beatles']

In [46]:
recommendations_1b = [a for a in similar_user_listened if a not in user_1606_listened]

print(f"RECOMMENDATIONS FOR USER 1606")
print(f"Total new recommendations: {len(recommendations_1b)}")
print(recommendations_1b)

RECOMMENDATIONS FOR USER 1606
Total new recommendations: 11
['beastie boys', 'bob dylan', 'bob marley & the wailers', 'david bowie', 'eric clapton', 'johnny cash', 'pearl jam', 'pink floyd', 'the doors', 'the rolling stones', 'tom waits']


#### **C. How many recommended songs has user 1606 already heard?**
Finding the intersection between recommendations and user 1606's history

##### **Convert both lists to sets and find the overlap**

In [47]:
already_heard = set(recommendations_1b) & set(user_1606_listened)

print(f"Recommendations already heard by user 1606: {len(already_heard)}")
print(f"Artists: {already_heard}")

Recommendations already heard by user 1606: 0
Artists: set()


In [49]:
# Check overlap using User 1144's full list (before we removed user 1606's artists)
# This shows how many of User 1144's artists user 1606 already knew
already_heard_before_filter = set(similar_user_listened) & set(user_1606_listened)

print(f"Overlap before filtering: {len(already_heard_before_filter)}")
print(f"Artists already heard: {already_heard_before_filter}")

Overlap before filtering: 2
Artists already heard: {'the beatles', 'elvis presley'}


So far, we build recommendations based on artists that are listened to by the same group of people (item-item CF) and based on finding the user that have the similar artist list (user-user CF).

However:
- With item-item CF, we don't know anythign about the specific user we're recommending to. It is the same recommendation for everyont who likes u2.
- With user-use CF, we only look at one similar user's history and ignore item relationship entirely.

As a results, we combines both the CF types to look at how similar is this artist to what user 1606 already likes and weighted by how many people who listened to similar artists also listened to this one. 


#### **D. Use a combination of user-item approach to build a recommendation score for each song for each user**

##### **1. Get artists user 1606 already listened to**

In [50]:
user_1606_row = df.loc[1606]

In [51]:
listened = user_1606_row[user_1606_row > 0].index.tolist()
not_listened = user_1606_row[user_1606_row == 0].index.tolist()

In [53]:
print(f"User 1606 seed artists (already heard):")
print(listened)
print(f"Total candidate artists to score: {len(not_listened)}")

User 1606 seed artists (already heard):
['abba', 'elvis presley', 'frank sinatra', 'the beatles']
Total candidate artists to score: 280


There are 280 remimaning artists that user 1606 has not discovered yet, we need to score and rank to find the best recommendation.

##### **2. Compute recommendation score for each candidate**

In [54]:
scores = {}

for candidate in not_listened:
    
    # Get top 10 most similar artists to this candidate (from item_sim_matrix)
    top10_similar = item_sim_matrix[candidate].drop(candidate)\
                    .sort_values(ascending=False).head(10)
    
    # Get user 1606's listening history for those top 10 similar artists
    purchase_history = df.loc[1606, top10_similar.index]
    
    # Numerator: sum of (purchase history × similarity score)
    numerator = (purchase_history * top10_similar).sum()
    
    # Denominator: sum of all similarity scores
    denominator = top10_similar.sum()
    
    # Compute final score
    if denominator > 0:
        scores[candidate] = numerator / denominator
    else:
        scores[candidate] = 0

# Convert to series and sort highest to lowest
scores_series = pd.Series(scores).sort_values(ascending=False)

print("TOP 5 RECOMMENDATIONS FOR USER 1606 (HYBRID SCORE)")
print(scores_series.head(5))

TOP 5 RECOMMENDATIONS FOR USER 1606 (HYBRID SCORE)
eric clapton     0.202740
howard shore     0.171749
david bowie      0.171243
dream theater    0.134501
apocalyptica     0.106527
dtype: float64


The scores are relatively low (max 0.20) because user 1606 only has 4 seed artists, which means there is not much history to drive strong signals

### **IV. Neural Network using Numpy**
In the second notebook.